# Model Hyperparameter Search

This notebook is an example of how to run a hyperparameter search for the pyrul benchmark.

The chosen model example is the DAST, but based on the available set of models, this notebook can be easily modified.

In [ ]:
%reset -f

In [ ]:
import sys
sys.path.append('../')
import pyrul
import pandas as pd

In [ ]:
IDX = 0 # random internal seed

In [ ]:
# Path to save the database of HP search and study name
db_path = "sqlite:////rul_tuning_db.sqlite3"
study_name = "DAST_cmapss_tuning"

#data_load_path = "../datasets/cmapss/train_FD001_fixed.txt" # CMAPSS data
best_data_save_path = "./tmp_tuning_models/DAST_paper_processed"
best_data_metrics_save_path = "./tmp_tuning_models/"
best_model_save_path = "./tmp_tuning_models/DAST"

In [ ]:
SEED = pyrul.utils.seed.get_seed(IDX)
pyrul.utils.seed.set_seed(SEED)

In [ ]:
# Clearing tempory plots folder
pyrul.utils.file.clear_tmp_plots(tmp_plot_path)

In [ ]:
# Declaring the model and its hyperparameters that will not change
model = pyrul.models.dast.DAST_Model(
     DATASET_NAME = "cmapss_fd001",
     R_EARLY = 125,
     WINDOW = 30,
     SEED=SEED)

# The hyperparameters to search are defined inside the DAST code. They correspond in this case to:
# - the learning rate (1e-5, 3e-5, 6e-5, 1e-4, 3e-4, ..., 3e-1 or 6e-1)
# - the window of the decoder (last x values from the initial window) (between 5 and WINDOW//2 with a step of 5)
# - the number of attention heads (between 1 and 3)
# - the number of neurons in layers (16 or 32)
# - the number of encoding layers (between 1 and 4)
# - the number of decoding layers (between 1 and 4)

# Declaring the dataloader
dl = pyrul.models.data.DataLoader(dataset="cmapss_fd001")

# Declaring the evaluator and its metrics
evaluator = pyrul.evaluation.Evaluator(metrics=["rmse"])

In [ ]:
# To delete the previous search
# optuna.delete_study(study_name=study_name, storage=db_path)

In [ ]:
# Running the hyperparameter search

# In the case of DAST, we use Bayesian search with 30 trials, 10 of which are random guesses
mySampler = optuna.samplers.GPSampler(seed=SEED, n_startup_trials=10) # to start a study
mySampler = pickle.load(open("sampler.pkl", "rb")) # to resume study

model.run_hp_search(sampler=mySampler,
                    save_db_path=db_path, 
                    study_name=study_name,
                    dataloader=dl,
                    evaluator=evaluator,
                    data_load_path=data_load_path, 
                    best_data_save_path=best_data_save_path,
                    best_data_metrics_save_path=best_data_metrics_save_path,
                    best_model_save_path=best_model_save_path,
                    n_trials=30,
                    best_trial_idx=-1 # this index should be manually set to the best trial when resuming a study !
                    )

In [ ]:
# After the HP search, the best model, its data metrics and datasets are saved inside the "tmp_tuning_models" folder